In [1]:
import torch
from transformers import AutoModelForCausalLM
from deepseek_vl.models import VLChatProcessor, MultiModalityCausalLM
from deepseek_vl.utils.io import load_pil_images

# 本地模型路径
model_path = "/disk1/users/fengyy/projects/models/deepseek-vl-7b-chat"

# 加载 processor 和 tokenizer
vl_chat_processor: VLChatProcessor = VLChatProcessor.from_pretrained(model_path, local_files_only=True)
tokenizer = vl_chat_processor.tokenizer

# 加载模型
vl_gpt: MultiModalityCausalLM = AutoModelForCausalLM.from_pretrained(
    model_path, 
    trust_remote_code=True,
    local_files_only=True,  # 强制使用本地文件
    torch_dtype=torch.bfloat16,
    device_map="auto"  # 自动分配层到GPU，或去掉手动 .to('cuda:5')
)

# 如果不用 device_map，手动指定GPU
# vl_gpt = vl_gpt.to(torch.bfloat16).to('cuda:5').eval()

print(f"✓ 模型加载成功 from {model_path}")

/disk1/users/fengyy/.conda/envs/Vattack/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python version is above 3.10, patching the collections module.


/disk1/users/fengyy/.conda/envs/Vattack/lib/python3.10/site-packages/transformers/models/auto/image_processing_auto.py:647: FutureWarning: The image_processor_class argument is deprecated and will be removed in v4.42. Please use `slow_image_processor_class`, or `fast_image_processor_class` instead
  warnings.warn(
/disk1/users/fengyy/.conda/envs/Vattack/lib/python3.10/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/disk1/users/fengyy/.conda/envs/Vattack/lib/python3.10/site-packages/pydantic/_internal/_generate_s

✓ 模型加载成功 from /disk1/users/fengyy/projects/models/deepseek-vl-7b-chat


Dir

In [2]:
import os
import torch
import json
from glob import glob

adv_dir = "/disk1/users/fengyy/projects/V-Attack/adv_images/co/co_targeted_v/0_1_no_vision"
vqa_json_path = 'coco300_vqa_main.json'
output_json = "co/deepseekvl_1.json"  

with open(vqa_json_path, 'r') as f:
    vqa_data = json.load(f)

image_to_questions = {item['image']: item['vqa'] for item in vqa_data}

adv_files = sorted(glob(os.path.join(adv_dir, "*.png")))

results = []

for i, adv_path in enumerate(adv_files):
    if i == 300:
        break

    filename = os.path.basename(adv_path)
    # .replace(".png", ".jpg")

    questions = image_to_questions.get(filename, ["", "", ""])
    if len(questions) != 3:
        questions = questions[:3] + [""] * (3 - len(questions))

    conversation = [
        {
            "role": "User",
            "content": "<image_placeholder>Describe this image in one sentence.",
            "images": [adv_path]
        },
        {
            "role": "Assistant",
            "content": ""
        }
    ]
    
    pil_images = load_pil_images(conversation)
    prepare_inputs = vl_chat_processor(
        conversations=conversation,
        images=pil_images,
        force_batchify=True
    ).to(vl_gpt.device)
    
    inputs_embeds = vl_gpt.prepare_inputs_embeds(**prepare_inputs)
    
    outputs = vl_gpt.language_model.generate(
        inputs_embeds=inputs_embeds,
        attention_mask=prepare_inputs.attention_mask,
        pad_token_id=tokenizer.eos_token_id,
        bos_token_id=tokenizer.bos_token_id,
        eos_token_id=tokenizer.eos_token_id,
        max_new_tokens=512,
        do_sample=False,
        use_cache=True
    )
    
    adv_response_1 = tokenizer.decode(outputs[0].cpu().tolist(), skip_special_tokens=True)
    print(f'ADV [{filename}] Q1: {adv_response_1}')
    
    adv_responses = [adv_response_1]
    for i, question_text in enumerate(questions, start=2):

        conversation[0]["content"] = f"<image_placeholder>{question_text}"

        pil_images = load_pil_images(conversation)
        prepare_inputs = vl_chat_processor(
            conversations=conversation,
            images=pil_images,
            force_batchify=True
        ).to(vl_gpt.device)
        
        inputs_embeds = vl_gpt.prepare_inputs_embeds(**prepare_inputs)
        
        outputs = vl_gpt.language_model.generate(
            inputs_embeds=inputs_embeds,
            attention_mask=prepare_inputs.attention_mask,
            pad_token_id=tokenizer.eos_token_id,
            bos_token_id=tokenizer.bos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            max_new_tokens=512,
            do_sample=False,
            use_cache=True
        )
        
        adv_response = tokenizer.decode(outputs[0].cpu().tolist(), skip_special_tokens=True)
        print(f'ADV [{filename}] Q{i}: {adv_response}')
        
        adv_responses.append(adv_response)
    
    results.append({
        "filename": filename,
        "adversarial_response_1": adv_responses[0],
        "adversarial_response_2": adv_responses[1] if len(adv_responses) > 1 else "",
        "adversarial_response_3": adv_responses[2] if len(adv_responses) > 2 else "",
        "adversarial_response_4": adv_responses[3] if len(adv_responses) > 3 else ""
    })

with open(output_json, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=4)

print(f"Results saved to {output_json}")

ADV [000000000285.png] Q1: A close-up of a lion sculpture in a grassy area.
ADV [000000000285.png] Q2: The image is a close-up photograph of a bear. The bear's fur appears to be light brown or tan in color, and its facial features are prominently displayed. The bear's eyes are open, and it seems to be looking directly at the camera. Its ears are perked up, and its mouth is slightly ajar, revealing its teeth. The bear's nose is black, and there is a patch of darker fur around it. The background of the image is a blurred green, suggesting a natural outdoor setting, possibly grass or foliage. There are no visible texts or discernible brands in the image. The style of the image is realistic, capturing the bear in a naturalistic manner.


KeyboardInterrupt: 

Dirs

In [ ]:
import os
import torch
import json
from glob import glob

adv_dirs = [
    '...',
    # Add more directories as needed
]
vqa_json_path = "coco300_vqa_main.json"
output_json_dir = "deepseekvl"  

with open(vqa_json_path, 'r') as f:
    vqa_data = json.load(f)

image_to_questions = {item['image']: item['vqa'] for item in vqa_data}

def process_directory(adv_dir):

    adv_files = sorted(glob(os.path.join(adv_dir, "*.png")) + glob(os.path.join(adv_dir, "*.jpg")))

    results = []

    for i, adv_path in enumerate(adv_files):

        filename = os.path.basename(adv_path)

        questions = image_to_questions.get(filename, ["", "", ""])
        if len(questions) != 3:
            questions = questions[:3] + [""] * (3 - len(questions))

        conversation = [
            {
                "role": "User",
                "content": "<image_placeholder>Describe this image in one sentence.",
                "images": [adv_path]
            },
            {
                "role": "Assistant",
                "content": ""
            }
        ]

        pil_images = load_pil_images(conversation)
        prepare_inputs = vl_chat_processor(
            conversations=conversation,
            images=pil_images,
            force_batchify=True
        ).to(vl_gpt.device)
        
        inputs_embeds = vl_gpt.prepare_inputs_embeds(**prepare_inputs)
        
        outputs = vl_gpt.language_model.generate(
            inputs_embeds=inputs_embeds,
            attention_mask=prepare_inputs.attention_mask,
            pad_token_id=tokenizer.eos_token_id,
            bos_token_id=tokenizer.bos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            max_new_tokens=512,
            do_sample=False,
            use_cache=True
        )
        
        adv_response_1 = tokenizer.decode(outputs[0].cpu().tolist(), skip_special_tokens=True)
        print(f'ADV [{filename}] Q1: {adv_response_1}')
        
        adv_responses = [adv_response_1]
        for i, question_text in enumerate(questions, start=2):

            conversation[0]["content"] = f"<image_placeholder>{question_text}"

            pil_images = load_pil_images(conversation)
            prepare_inputs = vl_chat_processor(
                conversations=conversation,
                images=pil_images,
                force_batchify=True
            ).to(vl_gpt.device)
            
            inputs_embeds = vl_gpt.prepare_inputs_embeds(**prepare_inputs)
            
            outputs = vl_gpt.language_model.generate(
                inputs_embeds=inputs_embeds,
                attention_mask=prepare_inputs.attention_mask,
                pad_token_id=tokenizer.eos_token_id,
                bos_token_id=tokenizer.bos_token_id,
                eos_token_id=tokenizer.eos_token_id,
                max_new_tokens=512,
                do_sample=False,
                use_cache=True
            )
            
            adv_response = tokenizer.decode(outputs[0].cpu().tolist(), skip_special_tokens=True)
            print(f'ADV [{filename}] Q{i}: {adv_response}')
            
            adv_responses.append(adv_response)
        
        results.append({
            "filename": filename,
            "adversarial_response_1": adv_responses[0],
            "adversarial_response_2": adv_responses[1] if len(adv_responses) > 1 else "",
            "adversarial_response_3": adv_responses[2] if len(adv_responses) > 2 else "",
            "adversarial_response_4": adv_responses[3] if len(adv_responses) > 3 else ""
        })
    
    if 'mattack' in adv_dir:
        output_json = os.path.join(output_json_dir, os.path.basename(adv_dir) + "_mattack_result.json")
    else:
        if 'ENS' in adv_dir:
            if 'targeted_v' in adv_dir:
                output_json = os.path.join(output_json_dir, os.path.basename(adv_dir) + "_ens_v_result.json")
            else:
                output_json = os.path.join(output_json_dir, os.path.basename(adv_dir) + "_ens_x_result.json")
            
        else:
            if 'targeted_v' in adv_dir:
                output_json = os.path.join(output_json_dir, os.path.basename(adv_dir) + "_sgl_v_result.json")
            else:
                output_json = os.path.join(output_json_dir, os.path.basename(adv_dir) + "_sgl_x_result.json")
    # output_json = os.path.join(output_json_dir, os.path.basename(adv_dir) + "_deepseek_result.json")
    with open(output_json, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=4)
    print(f"Results for {adv_dir} saved to {output_json}")

for adv_dir in adv_dirs:
    process_directory(adv_dir)
